# CC219 – Aplicaciones de Data Science · Trabajo Final
## Modelización: reseñas de videojuegos en Steam con Minería de Textos

**Integrantes:**
- Wilber Franz Toledo Mamani – u202320608
- Luis Enrique Aguilar Benites – u202219719
- Juan Miguel Quijano Calderon – u202221068

**Repositorio:** https://github.com/Lisztzz/CC219-TP-TF-2026-2-CC92

Este notebook es la **segunda fase** del proyecto. La primera (`TP_CC219_Steam_EDA.ipynb`) cargó, limpió y
exploró las 730 945 reseñas originales. Aquí se entrenan, comparan y explican los modelos que responden las
preguntas del caso de uso.

| # | Pregunta | Técnica | Sección |
|---|----------|---------|---------|
| P1 | ¿La reseña recomienda el juego? | Clasificación de texto: Naive Bayes, Regresión Logística, SVM y DistilBERT | 3 |
| P2 | ¿La reseña será valorada como útil? | Clasificación con texto + variables numéricas: Regresión Logística y LightGBM | 4 |
| P3 | ¿De qué tema trata la reseña? | Modelado de tópicos (NMF vs. LDA) + clasificador multiclase | 5 |
| E1 | ¿Qué juegos recomendar a quien le gustó uno? | Sistema de recomendación basado en contenido (NLP) con explicación | 6 |
| E2 | ¿La reseña es incoherente (texto vs. recomendación)? | Comparación VADER vs. modelo entrenado | 7 |
| E3 | ¿La reseña será considerada graciosa? | Clasificación de texto desbalanceada | 8 |

**Datos de entrada:** muestra estratificada de **50 000 reseñas** del dataset final (`data/steam_reviews_clean_sample.zip`
en el repositorio). Se usa la muestra para que cualquier persona pueda reproducir los resultados sin repetir la limpieza
de 700 mil reseñas y para que el ajuste fino de DistilBERT sea viable en una GPU gratuita de Colab.

**Cómo ejecutarlo:** en Google Colab, *Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)* y luego
*Ejecutar todas*. Sin GPU todo funciona igual, excepto la sección 3.6 (DistilBERT), que se omite.

## 1. Librerías y configuración

In [ ]:
# Instalamos versiones fijas para que los resultados sean reproducibles
!pip install -q scikit-learn==1.9.1 lightgbm==4.7.0 gensim==4.4.0 lime vaderSentiment

In [ ]:
# Librerias basicas
import os, time, warnings
import numpy as np
import pandas as pd
import joblib

# Graficos
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning clasico (scikit-learn)
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, cross_val_predict
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import ComplementNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.decomposition import NMF, LatentDirichletAllocation
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             average_precision_score, confusion_matrix, ConfusionMatrixDisplay,
                             roc_curve, precision_recall_curve, classification_report)
from sklearn.metrics.pairwise import cosine_similarity

# Otros modelos y herramientas
import lightgbm as lgb                                     # gradient boosting (P2)
from gensim.corpora import Dictionary                      # coherencia de topicos (P3)
from gensim.models.coherencemodel import CoherenceModel
from lime.lime_text import LimeTextExplainer               # explicabilidad (XAI)
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer   # sentimiento lexico (E2)


# Configuracion general
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 150)
pd.set_option('display.float_format', '{:.3f}'.format)

RANDOM_STATE = 42          # semilla: mismos resultados en cada ejecucion
N_JOBS = -1                # usar todos los nucleos del procesador

# Paleta fija (la misma del notebook de EDA)
C_POS, C_NEG, C_MAIN, C_GRAY = '#2a78d6', '#eb6834', '#2a78d6', '#8a8984'
PALETA_MODELOS = ['#2a78d6', '#eb6834', '#1a9e77', '#7b5ea7', '#c9a227']

sns.set_theme(style='whitegrid', rc={'axes.spines.top': False, 'axes.spines.right': False,
                                     'grid.color': '#e6e5e1', 'axes.titleweight': 'bold'})

## 2. Carga y preparación de los datos
Se lee la muestra del dataset limpio desde el repositorio. Si el notebook no se ejecuta dentro del repositorio
(por ejemplo, en Colab), primero se clona el repositorio de GitHub: así se obtienen los datos y el módulo
`code/app/preprocesamiento.py`, que contiene la misma limpieza de texto del notebook de EDA.

In [ ]:
import sys

# Buscamos la raiz del repositorio: la carpeta actual, la carpeta padre (si estamos en code/) o un clon nuevo
REPO_URL = 'https://github.com/Lisztzz/CC219-TP-TF-2026-2-CC92.git'
candidatas = ['.', '..', 'CC219-TP-TF-2026-2-CC92']
BASE = next((c for c in candidatas if os.path.exists(os.path.join(c, 'code', 'app', 'preprocesamiento.py'))), None)

if BASE is None:
    # No estamos dentro del repositorio: lo clonamos (solo la ultima version)
    !git clone -q --depth 1 {REPO_URL}
    BASE = 'CC219-TP-TF-2026-2-CC92'

# Modulo de limpieza de texto (el mismo del notebook de EDA)
sys.path.append(os.path.join(BASE, 'code', 'app'))
from preprocesamiento import preparar

# Carpetas de salida dentro de code/: figuras (informe), resultados (tablas) y modelos (app)
DIR_FIGURAS = os.path.join(BASE, 'code', 'figuras')
DIR_RESULTADOS = os.path.join(BASE, 'code', 'resultados')
DIR_MODELOS = os.path.join(BASE, 'code', 'app', 'modelos')
for carpeta in [DIR_FIGURAS, DIR_RESULTADOS, DIR_MODELOS]:
    os.makedirs(carpeta, exist_ok=True)


def guardar_figura(nombre):
    # Guarda la figura actual (para el informe y la presentacion)
    plt.savefig(os.path.join(DIR_FIGURAS, f'{nombre}.png'), dpi=150, bbox_inches='tight')


ruta = os.path.join(BASE, 'data', 'steam_reviews_clean_sample.zip')
print('Leyendo:', ruta)

# pandas descomprime el .zip automaticamente
df = pd.read_csv(ruta)
print('Dimensiones:', df.shape)
df.head(3)

In [ ]:
# Textos vacios -> cadena vacia (evita errores en el vectorizador)
df['review_clean'] = df['review_clean'].fillna('')
df['review_norm'] = df['review_norm'].fillna('')

# Variables objetivo como enteros (1 = si, 0 = no)
df['recomienda'] = df['voted_up'].astype(int)     # P1
# helpful (P2) y funny (E3) ya vienen como 0/1 desde el notebook de EDA

# Variables numericas en escala logaritmica (reducen el efecto de valores extremos)
df['log_palabras'] = np.log1p(df['word_count'])
df['log_horas'] = np.log1p(df['horas_jugadas'])
df['log_antiguedad'] = np.log1p(df['antiguedad_dias'])

# Proporcion de cada clase objetivo
pd.DataFrame({
    'P1 recomienda': df['recomienda'].mean(),
    'P2 helpful': df['helpful'].mean(),
    'E3 funny': df['funny'].mean(),
}, index=['% clase positiva']).T

### 2.1 Funciones de evaluación
Todas las preguntas se evalúan con las mismas funciones, así las tablas son comparables.

In [ ]:
def puntajes(modelo, X):
    # Devuelve un puntaje continuo por fila: probabilidad si existe, si no la distancia al hiperplano (SVM)
    if hasattr(modelo, 'predict_proba'):
        return modelo.predict_proba(X)[:, 1]
    return modelo.decision_function(X)


def evaluar(nombre, y_real, y_pred, y_puntaje, extra=None):
    # Calcula las metricas de un clasificador binario y las devuelve como diccionario
    fila = {
        'Modelo': nombre,
        'Accuracy': accuracy_score(y_real, y_pred),
        'Precision (clase 1)': precision_score(y_real, y_pred, zero_division=0),
        'Recall (clase 1)': recall_score(y_real, y_pred),
        'F1 (clase 1)': f1_score(y_real, y_pred),
        'F1 (clase 0)': f1_score(y_real, y_pred, pos_label=0),
        'F1 macro': f1_score(y_real, y_pred, average='macro'),
        'AUC-ROC': roc_auc_score(y_real, y_puntaje),
        'AUC-PR': average_precision_score(y_real, y_puntaje),
    }
    if extra:
        fila.update(extra)
    return fila


def tfidf(max_features=100_000):
    # Vectorizador TF-IDF comun: palabras y bigramas, ignora terminos que aparecen en menos de 3 reseñas
    return TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=max_features, sublinear_tf=True)


def curva_roc(ax, y_real, y_puntaje, nombre, color):
    # Dibuja la curva ROC de un modelo con su AUC en la leyenda
    fpr, tpr, _ = roc_curve(y_real, y_puntaje)
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{nombre} (AUC = {roc_auc_score(y_real, y_puntaje):.3f})')


def curva_pr(ax, y_real, y_puntaje, nombre, color):
    # Dibuja la curva precision-recall de un modelo con su AUC-PR en la leyenda
    precision, recall, _ = precision_recall_curve(y_real, y_puntaje)
    ax.plot(recall, precision, color=color, lw=2, label=f'{nombre} (AUC-PR = {average_precision_score(y_real, y_puntaje):.3f})')


# Validacion cruzada estratificada de 5 folds (la misma para todas las preguntas)
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

## 3. P1 – ¿La reseña recomienda el juego?
Clasificación binaria del texto limpio (`review_clean`). Se comparan tres algoritmos clásicos sobre TF-IDF y, con GPU,
un transformer preentrenado (DistilBERT) sobre el texto natural (`review_norm`).

- **Complement Naive Bayes:** modelo probabilístico rápido; la variante *Complement* está pensada para clases desbalanceadas.
- **Regresión Logística:** modelo lineal interpretable (cada palabra tiene un peso).
- **SVM lineal:** busca la frontera de máximo margen entre clases; suele ser el más fuerte en texto con TF-IDF.

Los modelos lineales usan `class_weight='balanced'` para compensar el desbalance (81 % positivas).

### 3.1 Partición entrenamiento / prueba

In [ ]:
# 80 % entrenamiento y 20 % prueba, manteniendo la proporcion de clases (stratify)
train_p1, test_p1 = train_test_split(df, test_size=0.2, stratify=df['recomienda'], random_state=RANDOM_STATE)

X_train, y_train = train_p1['review_clean'], train_p1['recomienda']
X_test, y_test = test_p1['review_clean'], test_p1['recomienda']

print('Entrenamiento:', len(X_train), '| Prueba:', len(X_test))

### 3.2 Validación cruzada de 5 folds (solo con el conjunto de entrenamiento)

In [ ]:
%%time
# Los tres modelos como pipelines: TF-IDF + clasificador
modelos_p1 = {
    'Naive Bayes (Complement)': make_pipeline(tfidf(), ComplementNB()),
    'Regresión Logística':      make_pipeline(tfidf(), LogisticRegression(max_iter=2000, class_weight='balanced')),
    'SVM lineal':               make_pipeline(tfidf(), LinearSVC(C=0.5, class_weight='balanced')),
}

filas_cv = []
for nombre, modelo in modelos_p1.items():
    # cross_validate entrena 5 veces con 4 folds y evalua con el fold restante
    r = cross_validate(modelo, X_train, y_train, cv=CV, scoring=['f1_macro', 'roc_auc'], n_jobs=N_JOBS)
    filas_cv.append({
        'Modelo': nombre,
        'F1 macro (media CV)': r['test_f1_macro'].mean(),
        'F1 macro (desv. CV)': r['test_f1_macro'].std(),
        'AUC-ROC (media CV)': r['test_roc_auc'].mean(),
    })

tabla_cv_p1 = pd.DataFrame(filas_cv).set_index('Modelo')
tabla_cv_p1

### 3.3 Evaluación en el conjunto de prueba

In [ ]:
%%time
# Entrenamos cada modelo con todo el conjunto de entrenamiento y evaluamos en prueba
resultados_p1 = []
puntajes_p1 = {}

for nombre, modelo in modelos_p1.items():
    inicio = time.time()
    modelo.fit(X_train, y_train)
    segundos = time.time() - inicio

    y_pred = modelo.predict(X_test)
    puntajes_p1[nombre] = puntajes(modelo, X_test)
    resultados_p1.append(evaluar(nombre, y_test, y_pred, puntajes_p1[nombre], {'Entrenamiento (s)': segundos}))

tabla_p1 = pd.DataFrame(resultados_p1).set_index('Modelo')
tabla_p1

In [ ]:
# Curvas ROC de los tres modelos y matriz de confusion del mejor (mayor F1 macro)
mejor_p1 = tabla_p1['F1 macro'].idxmax()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Izquierda: curvas ROC
for (nombre, s), color in zip(puntajes_p1.items(), PALETA_MODELOS):
    curva_roc(axes[0], y_test, s, nombre, color)
axes[0].plot([0, 1], [0, 1], ls='--', color=C_GRAY, lw=1)
axes[0].legend(loc='lower right')
axes[0].set_title('P1 – Curvas ROC en el conjunto de prueba')
axes[0].set_xlabel('Tasa de falsos positivos')
axes[0].set_ylabel('Tasa de verdaderos positivos')

# Derecha: matriz de confusion del mejor modelo
ConfusionMatrixDisplay.from_predictions(
    y_test, modelos_p1[mejor_p1].predict(X_test),
    display_labels=['No recomienda', 'Recomienda'], cmap='Blues', colorbar=False, ax=axes[1],
)
axes[1].set_title(f'P1 – Matriz de confusión: {mejor_p1}')
axes[1].set_xlabel('Predicción')
axes[1].set_ylabel('Valor real')
axes[1].grid(False)

plt.tight_layout()
guardar_figura('p1_roc_confusion')
plt.show()

### 3.4 Explicabilidad (XAI): ¿qué palabras usa el modelo?
La Regresión Logística asigna un **coeficiente** a cada término: los negativos empujan la predicción hacia
“no recomienda” y los positivos hacia “recomienda”.

In [ ]:
# Coeficientes de la Regresion Logistica
modelo_lr_p1 = modelos_p1['Regresión Logística']
terminos = modelo_lr_p1.named_steps['tfidfvectorizer'].get_feature_names_out()
coeficientes = pd.Series(modelo_lr_p1.named_steps['logisticregression'].coef_[0], index=terminos)

top_neg = coeficientes.nsmallest(15)
top_pos = coeficientes.nlargest(15)


fig, axes = plt.subplots(1, 2, figsize=(13, 6))

axes[0].barh(top_neg.index[::-1], top_neg.values[::-1], color=C_NEG, height=0.6)
axes[0].set_title('Términos que empujan a "no recomienda"')
axes[0].set_xlabel('Coeficiente')

axes[1].barh(top_pos.index[::-1], top_pos.values[::-1], color=C_POS, height=0.6)
axes[1].set_title('Términos que empujan a "recomienda"')
axes[1].set_xlabel('Coeficiente')

plt.tight_layout()
guardar_figura('p1_coeficientes')
plt.show()

**LIME** explica una predicción individual: modifica la reseña quitando palabras y observa cómo cambia la
probabilidad. Así se ve qué palabras pesaron en *esa* reseña concreta.

In [ ]:
# Explicamos con LIME la prediccion de una reseña negativa del conjunto de prueba
explicador = LimeTextExplainer(class_names=['No recomienda', 'Recomienda'], random_state=RANDOM_STATE)

# Elegimos una reseña negativa de longitud media (facil de leer)
candidatas = test_p1[(test_p1['recomienda'] == 0) & test_p1['n_palabras_clean'].between(15, 30)]
ejemplo = candidatas.iloc[0]

print('Reseña original:', ejemplo['review'][:300])
print('Texto limpio   :', ejemplo['review_clean'])
print('Prob. recomienda según el modelo:', round(modelo_lr_p1.predict_proba([ejemplo['review_clean']])[0, 1], 3))

explicacion = explicador.explain_instance(ejemplo['review_clean'], modelo_lr_p1.predict_proba, num_features=8)

# Grafico de LIME: barras naranjas empujan a "no recomienda", azules a "recomienda"
pesos = pd.Series(dict(explicacion.as_list())).sort_values()
plt.figure(figsize=(8, 4.5))
plt.barh(pesos.index, pesos.values, color=[C_NEG if v < 0 else C_POS for v in pesos.values], height=0.6)
plt.axvline(0, color=C_GRAY, lw=1)
plt.title('LIME – palabras que explican la predicción de esta reseña')
plt.xlabel('Peso en la probabilidad de "recomienda"')
plt.tight_layout()
guardar_figura('p1_lime')
plt.show()

### 3.5 Experimento con reseñas nuevas (escritas por nosotros)

In [ ]:
# Reseñas que el modelo nunca vio, incluyendo negaciones y un caso sarcastico
reseñas_nuevas = [
    'Amazing story and great soundtrack, one of the best games I have ever played.',
    'Not worth the price. The game crashes every ten minutes, I want a refund.',
    'It is not bad at all, actually pretty fun with friends.',
    'Wow, another update that breaks everything. Great job devs, truly a masterpiece.',
]

# Aplicamos la misma limpieza del notebook de EDA (funcion preparar del modulo preprocesamiento.py)
limpias = [preparar(t)[1] for t in reseñas_nuevas]
pd.DataFrame({
    'Reseña': reseñas_nuevas,
    'Prob. recomienda (Reg. Logística)': modelo_lr_p1.predict_proba(limpias)[:, 1].round(3),
    'Predicción SVM': np.where(modelos_p1['SVM lineal'].predict(limpias) == 1, 'Recomienda', 'No recomienda'),
})

### 3.6 DistilBERT (transformer preentrenado) – requiere GPU
**DistilBERT** es una versión compacta de BERT preentrenada con millones de textos en inglés. Aquí se hace *ajuste fino*
(fine-tuning): se entrena una época más con nuestras reseñas para que aprenda a clasificar la recomendación. A diferencia
de TF-IDF, lee la reseña completa en orden (`review_norm`, sin quitar stopwords), por lo que puede captar contexto,
negaciones y parte del sarcasmo. Usa la **misma partición** de prueba que los modelos clásicos y pesos de clase.

Si no hay GPU disponible, esta sección se omite y la tabla final muestra solo los modelos clásicos.

In [ ]:
# Intentamos usar la GPU; sin GPU esta seccion se salta (en CPU tardaria horas)
try:
    import torch
    USAR_DISTILBERT = torch.cuda.is_available()
except ImportError:          # entorno sin PyTorch
    USAR_DISTILBERT = False

print('GPU disponible:', USAR_DISTILBERT)
if USAR_DISTILBERT:
    print('Tarjeta:', torch.cuda.get_device_name(0))

In [ ]:
%%time
if USAR_DISTILBERT:
    !pip install -q datasets
    from datasets import Dataset
    from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                              TrainingArguments, Trainer, DataCollatorWithPadding)

    MODELO_BASE = 'distilbert-base-uncased'
    tokenizador = AutoTokenizer.from_pretrained(MODELO_BASE)

    # Datasets de Hugging Face con el texto natural (review_norm) y la etiqueta
    def a_dataset(datos):
        return Dataset.from_dict({'text': datos['review_norm'].tolist(), 'label': datos['recomienda'].tolist()})

    def tokenizar(lote):
        # Recortamos a 128 tokens: cubre la gran mayoria de reseñas (mediana de 22 palabras)
        return tokenizador(lote['text'], truncation=True, max_length=128)

    ds_train = a_dataset(train_p1).map(tokenizar, batched=True, remove_columns=['text'])
    ds_test = a_dataset(test_p1).map(tokenizar, batched=True, remove_columns=['text'])

    # Pesos de clase: equivocarse en una negativa cuesta mas (misma idea que class_weight='balanced')
    proporcion = train_p1['recomienda'].value_counts(normalize=True).sort_index()
    pesos_clase = torch.tensor((1 / (2 * proporcion)).values, dtype=torch.float)

    class TrainerPonderado(Trainer):
        # Trainer con funcion de perdida ponderada por clase
        def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
            etiquetas = inputs.pop('labels')
            salida = model(**inputs)
            perdida = torch.nn.functional.cross_entropy(salida.logits, etiquetas,
                                                        weight=pesos_clase.to(salida.logits.device))
            return (perdida, salida) if return_outputs else perdida

    modelo_bert = AutoModelForSequenceClassification.from_pretrained(MODELO_BASE, num_labels=2)

    argumentos = TrainingArguments(
        output_dir='distilbert_p1',
        num_train_epochs=1,                 # una epoca basta para ajustar un modelo preentrenado
        per_device_train_batch_size=32,
        per_device_eval_batch_size=128,
        learning_rate=2e-5,
        weight_decay=0.01,
        warmup_steps=100,
        fp16=True,                          # precision mixta: mas rapido en GPU
        logging_steps=200,
        save_strategy='no',
        report_to='none',
        seed=RANDOM_STATE,
    )

    entrenador = TrainerPonderado(model=modelo_bert, args=argumentos, train_dataset=ds_train,
                                  data_collator=DataCollatorWithPadding(tokenizador))
    inicio = time.time()
    entrenador.train()
    segundos_bert = time.time() - inicio

    # Prediccion en el conjunto de prueba: probabilidad de "recomienda" con softmax
    logits = entrenador.predict(ds_test).predictions
    proba_bert = torch.softmax(torch.tensor(logits), dim=1)[:, 1].numpy()
    pred_bert = (proba_bert >= 0.5).astype(int)

    puntajes_p1['DistilBERT (fine-tuning)'] = proba_bert
    resultados_p1.append(evaluar('DistilBERT (fine-tuning)', y_test, pred_bert, proba_bert,
                                 {'Entrenamiento (s)': segundos_bert}))
    tabla_p1 = pd.DataFrame(resultados_p1).set_index('Modelo')

    # Las mismas reseñas nuevas de la seccion 3.5
    entradas = tokenizador([preparar(t)[0] for t in reseñas_nuevas], truncation=True, max_length=128,
                           padding=True, return_tensors='pt').to(modelo_bert.device)
    with torch.no_grad():
        prob_nuevas = torch.softmax(modelo_bert(**entradas).logits, dim=1)[:, 1].cpu().numpy()
    print(pd.DataFrame({'Reseña': reseñas_nuevas, 'Prob. recomienda (DistilBERT)': prob_nuevas.round(3)}))
else:
    print('Sin GPU: se omite DistilBERT. En Colab: Entorno de ejecucion -> Cambiar tipo de entorno -> GPU.')

### 3.7 Tabla comparativa final de P1

In [ ]:
# Tabla final ordenada por F1 macro (incluye DistilBERT si se entreno)
columnas = ['Accuracy', 'F1 (clase 0)', 'F1 (clase 1)', 'F1 macro', 'AUC-ROC', 'AUC-PR', 'Entrenamiento (s)']
tabla_p1 = tabla_p1.sort_values('F1 macro', ascending=False)
tabla_p1.to_csv(f'{DIR_RESULTADOS}/p1_resultados.csv')
tabla_p1[columnas]

In [ ]:
# Grafico comparativo de F1 macro y AUC-ROC por modelo
comparacion = tabla_p1[['F1 macro', 'AUC-ROC']].sort_values('F1 macro')

fig, ax = plt.subplots(figsize=(9, 0.9 * len(comparacion) + 1.5))
posiciones = np.arange(len(comparacion))
ax.barh(posiciones - 0.2, comparacion['F1 macro'], height=0.38, color=C_POS, label='F1 macro')
ax.barh(posiciones + 0.2, comparacion['AUC-ROC'], height=0.38, color=C_NEG, label='AUC-ROC')

for p, (f1, auc) in zip(posiciones, comparacion.values):
    ax.text(f1 + 0.005, p - 0.2, f'{f1:.3f}', va='center', fontsize=9)
    ax.text(auc + 0.005, p + 0.2, f'{auc:.3f}', va='center', fontsize=9)

ax.set_yticks(posiciones)
ax.set_yticklabels(comparacion.index)
ax.set_xlim(0.5, 1.0)
ax.set_title('P1 – Comparación de modelos (conjunto de prueba)')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=2, frameon=False)
plt.tight_layout()
guardar_figura('p1_comparacion')
plt.show()

## 4. P2 – ¿La reseña será valorada como útil?
Objetivo: `helpful` = 1 si la reseña recibió 2 o más votos de “útil” (18.4 % de las reseñas). En el EDA se vio que los
votos dependen de la antigüedad, la longitud y el sentimiento, así que se comparan tres fuentes de información:

1. **Solo texto** (TF-IDF): ¿lo que dice la reseña predice su utilidad?
2. **Solo variables numéricas**: longitud, horas jugadas, si recomienda y precio, con y sin la **antigüedad** (variable de control).
3. **Texto + numéricas** combinados.

Como la clase útil es minoritaria, la métrica principal es el **AUC-PR** (un modelo al azar obtendría 0.184).

In [ ]:
# Variables numericas candidatas
NUM_SIN_ANTIG = ['log_palabras', 'log_horas', 'recomienda', 'precio_usd']
NUM_CON_ANTIG = NUM_SIN_ANTIG + ['log_antiguedad']

# Particion 80/20 estratificada por la clase util
train_p2, test_p2 = train_test_split(df, test_size=0.2, stratify=df['helpful'], random_state=RANDOM_STATE)
y_train_p2, y_test_p2 = train_p2['helpful'], test_p2['helpful']


def lgbm():
    # LightGBM con pesos de clase balanceados
    return lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31, class_weight='balanced',
                              random_state=RANDOM_STATE, verbose=-1)


def texto_y_numericas(columnas_num):
    # Une TF-IDF del texto con las variables numericas estandarizadas
    return ColumnTransformer([('texto', tfidf(50_000), 'review_clean'), ('num', StandardScaler(), columnas_num)])


modelos_p2 = {
    'Reg. Logística – solo texto':                 (make_pipeline(tfidf(50_000), LogisticRegression(max_iter=2000, class_weight='balanced')), 'review_clean'),
    'LightGBM – numéricas sin antigüedad':          (lgbm(), NUM_SIN_ANTIG),
    'LightGBM – numéricas con antigüedad':          (lgbm(), NUM_CON_ANTIG),
    'Reg. Logística – texto + numéricas':           (Pipeline([('cols', texto_y_numericas(NUM_CON_ANTIG)), ('lr', LogisticRegression(max_iter=3000, class_weight='balanced'))]), None),
    'LightGBM – texto + numéricas':                 (Pipeline([('cols', texto_y_numericas(NUM_CON_ANTIG)), ('lgbm', lgb.LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=63, colsample_bytree=0.3, class_weight='balanced', random_state=RANDOM_STATE, verbose=-1))]), None),
}

In [ ]:
%%time
# Entrenamiento y evaluacion de cada combinacion
resultados_p2 = []
puntajes_p2 = {}

for nombre, (modelo, columnas_x) in modelos_p2.items():
    # columnas_x = None significa que el modelo recibe el DataFrame completo (texto + numericas)
    X_tr = train_p2 if columnas_x is None else train_p2[columnas_x]
    X_te = test_p2 if columnas_x is None else test_p2[columnas_x]

    inicio = time.time()
    modelo.fit(X_tr, y_train_p2)
    segundos = time.time() - inicio

    puntajes_p2[nombre] = puntajes(modelo, X_te)
    resultados_p2.append(evaluar(nombre, y_test_p2, modelo.predict(X_te), puntajes_p2[nombre],
                                 {'Entrenamiento (s)': segundos}))

tabla_p2 = pd.DataFrame(resultados_p2).set_index('Modelo').sort_values('AUC-PR', ascending=False)
tabla_p2.to_csv(f'{DIR_RESULTADOS}/p2_resultados.csv')
tabla_p2[['AUC-PR', 'AUC-ROC', 'Precision (clase 1)', 'Recall (clase 1)', 'F1 (clase 1)', 'Entrenamiento (s)']]

In [ ]:
# Curvas precision-recall e importancia de variables del LightGBM con antiguedad
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Izquierda: curvas precision-recall
for (nombre, s), color in zip(puntajes_p2.items(), PALETA_MODELOS):
    curva_pr(axes[0], y_test_p2, s, nombre, color)
axes[0].axhline(y_test_p2.mean(), ls='--', color=C_GRAY, lw=1)
axes[0].text(0.62, y_test_p2.mean() + 0.01, 'modelo al azar', color=C_GRAY, fontsize=9)
axes[0].set_title('P2 – Curvas precisión-recall')
axes[0].set_xlabel('Recall')
axes[0].set_ylabel('Precisión')
axes[0].legend(fontsize=8)

# Derecha: importancia (ganancia) de cada variable numerica
modelo_num = modelos_p2['LightGBM – numéricas con antigüedad'][0]
nombres_legibles = {'log_palabras': 'Longitud (palabras)', 'log_horas': 'Horas jugadas', 'recomienda': 'Recomienda el juego',
                    'precio_usd': 'Precio', 'log_antiguedad': 'Antigüedad de la reseña'}
importancia = pd.Series(modelo_num.booster_.feature_importance('gain'), index=NUM_CON_ANTIG)
importancia = (importancia / importancia.sum() * 100).rename(nombres_legibles).sort_values()

axes[1].barh(importancia.index, importancia.values, color=C_MAIN, height=0.6)
for p, v in enumerate(importancia.values):
    axes[1].text(v + 0.5, p, f'{v:.0f} %', va='center', fontsize=9)
axes[1].set_title('P2 – Importancia de variables (LightGBM)')
axes[1].set_xlabel('% de la ganancia total')

plt.tight_layout()
guardar_figura('p2_pr_importancia')
plt.show()

**Interpretación:** el texto por sí solo predice la utilidad mejor que el azar, pero las variables de **contexto**
(si la reseña recomienda, su longitud, las horas jugadas y su antigüedad) predicen más. Combinar ambas fuentes no
mejora de forma clara: la utilidad depende más de *quién* escribe y *cuánto tiempo* lleva publicada la reseña que de las
palabras exactas. La antigüedad confirma el sesgo de exposición detectado en el EDA.

## 5. P3 – ¿De qué tema trata la reseña?
No hay una etiqueta de tema en los datos, así que primero se **descubren** los temas con modelado de tópicos
(aprendizaje no supervisado) y luego se entrena un **clasificador** que asigna el tema a reseñas nuevas.

- **NMF** (factorización de matrices no negativas) sobre TF-IDF.
- **LDA** (asignación latente de Dirichlet) sobre conteos de palabras.

Se elige el número de temas *k* con la **coherencia c_v**: mide si las palabras principales de cada tema suelen
aparecer juntas en las reseñas (más alto = temas más interpretables).

Los temas se aprenden con las reseñas de **8 palabras limpias o más** (unas 30 mil): una reseña como *“good game”*
no habla de ningún tema y solo agrega ruido.

In [ ]:
# Palabras muy generales que aparecen en todos los temas y no ayudan a distinguirlos
STOP_TEMAS = ['not', 'no', 'good', 'great', 'fun', 'really', 'like', 'time', 'buy', 'recommend', 'make', 'get', 'got',
              'thing', 'lot', 'bit', 'way', 'want', 'feel', 'think', 'know', 'say', 'even', 'still', 'also', 'well',
              'much', 'would', 'one', 'go', 'pretty', 'little', 'best', 'bad', 'love', 'worth', 'nice', 'amazing',
              'overall', 'don', 'didn', 'doesn', 'isn', 'hour', 'year', 'day', 'enjoy', 'better', 'new', 'need', 'try',
              'use', 'people', 'awesome', 'experience', 'actually', 'definitely', 'highly', 'absolutely', 'start', 'end',
              'come', 'look', 'sure', 'shit', 'fucking', 'fuck', 'cool', 'super', 'life', 'man', 'point', 'far', 'right',
              'bought', 'review', 'money']

# Solo reseñas con 8 o mas palabras limpias
df_temas = df[df['n_palabras_clean'] >= 8].copy()
docs = df_temas['review_clean']
print('Reseñas usadas para descubrir temas:', len(docs))

# Matrices de entrada: TF-IDF para NMF y conteos para LDA
vec_nmf = TfidfVectorizer(min_df=10, max_df=0.4, max_features=20_000, stop_words=STOP_TEMAS)
X_nmf = vec_nmf.fit_transform(docs)
vec_lda = CountVectorizer(min_df=10, max_df=0.4, max_features=20_000, stop_words=STOP_TEMAS)
X_lda = vec_lda.fit_transform(docs)

# Diccionario de gensim para calcular coherencia (muestra de 15 mil reseñas para que sea rapido)
textos_coh = [d.split() for d in docs.sample(15_000, random_state=RANDOM_STATE)]
diccionario = Dictionary(textos_coh)


def palabras_top(componentes, vocabulario, n=10):
    # Las n palabras con mas peso en cada tema
    return [[vocabulario[i] for i in fila.argsort()[::-1][:n]] for fila in componentes]


def coherencia(temas):
    # Coherencia c_v promedio de un conjunto de temas
    temas = [[w for w in t if w in diccionario.token2id] for t in temas]
    return CoherenceModel(topics=temas, texts=textos_coh, dictionary=diccionario, coherence='c_v', processes=1).get_coherence()

In [ ]:
%%time
# Probamos k = 6, 8 y 10 temas con ambos algoritmos (tarda unos minutos)
filas = []
modelos_temas = {}

for k in [6, 8, 10]:
    nmf = NMF(n_components=k, init='nndsvda', max_iter=400, random_state=RANDOM_STATE).fit(X_nmf)
    lda = LatentDirichletAllocation(n_components=k, learning_method='online', random_state=RANDOM_STATE, n_jobs=N_JOBS).fit(X_lda)
    modelos_temas[('NMF', k)] = nmf
    modelos_temas[('LDA', k)] = lda

    filas.append({'k': k,
                  'NMF': coherencia(palabras_top(nmf.components_, vec_nmf.get_feature_names_out())),
                  'LDA': coherencia(palabras_top(lda.components_, vec_lda.get_feature_names_out()))})

tabla_coh = pd.DataFrame(filas).set_index('k')
tabla_coh.to_csv(f'{DIR_RESULTADOS}/p3_coherencia.csv')
tabla_coh

In [ ]:
# Grafico de coherencia segun k
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(tabla_coh.index, tabla_coh['NMF'], marker='o', lw=2, ms=7, color=C_POS, label='NMF (TF-IDF)')
ax.plot(tabla_coh.index, tabla_coh['LDA'], marker='o', lw=2, ms=7, color=C_NEG, label='LDA (conteos)')
ax.set_xticks(tabla_coh.index)
ax.set_title('P3 – Coherencia c_v según el número de temas')
ax.set_xlabel('Número de temas (k)')
ax.set_ylabel('Coherencia c_v')
ax.legend()
plt.tight_layout()
guardar_figura('p3_coherencia')
plt.show()

# Regla de eleccion: el algoritmo con mayor coherencia y, dentro de el, el mayor k cuya coherencia
# este a menos de 0.02 de la mejor (temas mas especificos sin perder interpretabilidad)
algoritmo = tabla_coh.max().idxmax()
mejor_coh = tabla_coh[algoritmo].max()
K = int(tabla_coh.index[tabla_coh[algoritmo] >= mejor_coh - 0.02].max())
print(f'Configuración elegida: {algoritmo} con k = {K} (coherencia {tabla_coh.loc[K, algoritmo]:.3f}; la mejor fue {mejor_coh:.3f})')

In [ ]:
# Palabras principales de cada tema del modelo elegido
modelo_temas = modelos_temas[(algoritmo, K)]
vec_temas = vec_nmf if algoritmo == 'NMF' else vec_lda
X_temas = X_nmf if algoritmo == 'NMF' else X_lda

top_temas = palabras_top(modelo_temas.components_, vec_temas.get_feature_names_out(), n=10)

# Nombre de cada tema: se asigna por palabras clave (interpretacion humana de las palabras principales).
# Si un tema no coincide con ninguna regla, se nombra con sus 3 palabras principales.
REGLAS_NOMBRE = [
    ({'crash', 'fix', 'issue'},             'Problemas técnicos'),
    ({'paint', 'spare', 'average'},         'Plantilla de reseña (checklist)'),
    ({'mod', 'modding', 'workshop'},        'Mods y comunidad'),
    ({'friend', 'coop', 'laugh'},           'Jugar con amigos'),
    ({'puzzle', 'solve'},                   'Puzles, arte y música'),
    ({'campaign', 'multiplayer', 'single'}, 'Campaña y multijugador'),
    ({'open', 'explore', 'exploration'},    'Mundo abierto y exploración'),
    ({'story', 'ending', 'line'},           'Historia y personajes'),
    ({'enemy', 'weapon', 'combat'},         'Combate y diseño de niveles'),
    ({'graphic', 'soundtrack', 'atmosphere'}, 'Gráficos y ambientación'),
]


def nombrar(palabras, usados):
    # Primera regla cuyas palabras clave aparezcan entre las 6 principales del tema
    for claves, nombre in REGLAS_NOMBRE:
        if nombre not in usados and claves & set(palabras[:6]):
            return nombre
    return ' · '.join(palabras[:3])


nombres_temas = []
for palabras in top_temas:
    nombres_temas.append(nombrar(palabras, nombres_temas))

# Tema dominante de cada reseña = el de mayor peso
pesos_temas = modelo_temas.transform(X_temas)
df_temas['tema'] = pesos_temas.argmax(axis=1)
df_temas['nombre_tema'] = df_temas['tema'].map(dict(enumerate(nombres_temas)))

tabla_temas = pd.DataFrame({
    'Tema': nombres_temas,
    'Palabras principales': [', '.join(t) for t in top_temas],
    '% de reseñas': df_temas['tema'].value_counts(normalize=True).sort_index().values * 100,
})
tabla_temas.to_csv(f'{DIR_RESULTADOS}/p3_temas.csv', index=False)
tabla_temas

In [ ]:
# ¿De que hablan las reseñas negativas? % de reseñas negativas dentro de cada tema
por_tema = df_temas.groupby('nombre_tema').agg(reseñas=('tema', 'size'), pct_negativas=('recomienda', lambda s: (1 - s.mean()) * 100))
por_tema = por_tema.sort_values('pct_negativas')
promedio_neg = (1 - df_temas['recomienda'].mean()) * 100

fig, ax = plt.subplots(figsize=(10, 0.55 * K + 1.5))
colores = [C_NEG if v > promedio_neg else C_POS for v in por_tema['pct_negativas']]
ax.barh(por_tema.index, por_tema['pct_negativas'], color=colores, height=0.6)
ax.axvline(promedio_neg, ls='--', color=C_GRAY, lw=1)
ax.text(promedio_neg + 0.3, -0.6, f'promedio {promedio_neg:.1f} %', color=C_GRAY, fontsize=9)
for p, v in enumerate(por_tema['pct_negativas']):
    ax.text(v + 0.3, p, f'{v:.1f} %', va='center', fontsize=9)
ax.set_title('P3 – % de reseñas negativas por tema (naranja = sobre el promedio)')
ax.set_xlabel('% de reseñas que no recomiendan el juego')
plt.tight_layout()
guardar_figura('p3_temas_negativas')
plt.show()

### 5.1 Clasificador de temas
Con el tema dominante como etiqueta se entrena una Regresión Logística multiclase. Este clasificador es el que asigna
el tema a reseñas nuevas (por ejemplo, en la app) y su F1 macro indica qué tan separables son los temas descubiertos.

In [ ]:
# Particion 80/20 estratificada por tema
train_p3, test_p3 = train_test_split(df_temas, test_size=0.2, stratify=df_temas['tema'], random_state=RANDOM_STATE)

clasificador_temas = make_pipeline(tfidf(50_000), LogisticRegression(max_iter=3000, class_weight='balanced'))
clasificador_temas.fit(train_p3['review_clean'], train_p3['tema'])
pred_temas = clasificador_temas.predict(test_p3['review_clean'])

print(f"Accuracy: {accuracy_score(test_p3['tema'], pred_temas):.3f} | F1 macro: {f1_score(test_p3['tema'], pred_temas, average='macro'):.3f}")
print(classification_report(test_p3['tema'], pred_temas, target_names=nombres_temas, digits=3))

# Temas asignados a las reseñas nuevas de la seccion 3.5
pd.DataFrame({'Reseña': reseñas_nuevas,
              'Tema asignado': [nombres_temas[t] for t in clasificador_temas.predict(limpias)]})

## 6. E1 – Sistema de recomendación basado en contenido
Cada juego se representa por lo que dicen sus **reseñas positivas** (lo que sus jugadores elogian). Esos perfiles se
convierten en vectores TF-IDF y se recomiendan los juegos con mayor **similitud coseno**. La recomendación se explica
con los términos que más comparten ambos juegos (Explainable AI).

**Evaluación:** no hay historial de compras de usuarios, así que se usa una prueba de *auto-recuperación*: las reseñas
de cada juego se dividen en dos mitades; con la mitad A se busca entre todas las mitades B. Si el perfil captura la
identidad del juego, su propia mitad B debería salir primera (Hit@1) o entre las cinco primeras (Hit@5).

In [ ]:
# Reseñas positivas de juegos con al menos 20 reseñas positivas en la muestra
positivas = df[df['recomienda'] == 1]
conteo = positivas.groupby('name').size()
juegos_validos = conteo[conteo >= 20].index
positivas = positivas[positivas['name'].isin(juegos_validos)]
print('Juegos en el recomendador:', len(juegos_validos))

STOP_RECO = ['not', 'no', 'good', 'great', 'fun', 'really', 'like', 'time', 'buy', 'recommend', 'best', 'love', 'worth', 'amazing']


def vectorizador_reco():
    return TfidfVectorizer(min_df=3, max_df=0.5, sublinear_tf=True, stop_words=STOP_RECO)

In [ ]:
# Evaluacion por auto-recuperacion: mitad A vs mitad B de las reseñas de cada juego
mezcladas = positivas.sample(frac=1, random_state=RANDOM_STATE)
mezcladas['mitad'] = mezcladas.groupby('name').cumcount() % 2

perfil_a = mezcladas[mezcladas['mitad'] == 0].groupby('name')['review_clean'].apply(' '.join)
perfil_b = mezcladas[mezcladas['mitad'] == 1].groupby('name')['review_clean'].apply(' '.join).reindex(perfil_a.index)

vec_eval = vectorizador_reco().fit(pd.concat([perfil_a, perfil_b]))
similitud_ab = cosine_similarity(vec_eval.transform(perfil_a), vec_eval.transform(perfil_b))

# Posicion en la que aparece la mitad correcta (0 = primera)
posicion = (similitud_ab > np.diag(similitud_ab)[:, None]).sum(axis=1)
tabla_e1 = pd.DataFrame({'Métrica': ['Hit@1', 'Hit@5', 'Hit@10', 'MRR'],
                         'Valor': [(posicion < 1).mean(), (posicion < 5).mean(), (posicion < 10).mean(), (1 / (posicion + 1)).mean()]})
tabla_e1.to_csv(f'{DIR_RESULTADOS}/e1_evaluacion.csv', index=False)
print(f'Un recomendador al azar tendría Hit@1 = {1 / len(perfil_a):.4f}')
tabla_e1

In [ ]:
# Recomendador final: perfiles con todas las reseñas positivas de cada juego
perfiles = positivas.groupby('name')['review_clean'].apply(' '.join)
vec_reco = vectorizador_reco()
matriz_perfiles = vec_reco.fit_transform(perfiles)
similitud = cosine_similarity(matriz_perfiles)
nombres_juegos = perfiles.index
terminos_reco = np.array(vec_reco.get_feature_names_out())


def recomendar(juego, n=5, n_terminos=4):
    # Devuelve los n juegos mas parecidos y los terminos que comparten (explicacion)
    i = nombres_juegos.get_loc(juego)
    indices = similitud[i].argsort()[::-1]
    indices = [j for j in indices if j != i][:n]

    filas = []
    for j in indices:
        compartidos = matriz_perfiles[i].multiply(matriz_perfiles[j]).toarray().ravel()
        filas.append({'Juego recomendado': nombres_juegos[j],
                      'Similitud': similitud[i, j],
                      'Porque ambos mencionan': ', '.join(terminos_reco[compartidos.argsort()[::-1][:n_terminos]])})
    return pd.DataFrame(filas)


for juego in ['Portal 2', 'Stardew Valley', 'Hades', 'Euro Truck Simulator 2']:
    if juego in nombres_juegos:
        print(f'\nA quien le gustó {juego}, le recomendamos:')
        display(recomendar(juego))

## 7. E2 – ¿La reseña es incoherente?
Una reseña es **incoherente** cuando su texto dice lo contrario de lo que marcó el usuario (sarcasmo, humor, error o
una reseña mixta). Se comparan dos “jueces” del texto:

- **VADER:** diccionario de sentimiento genérico, sin entrenamiento (usado en el EDA).
- **Modelo P1 (Regresión Logística):** entrenado con reseñas de Steam. Para no evaluar una reseña con un modelo que
  la vio al entrenar, se usan predicciones *fuera del fold* (`cross_val_predict` con 5 folds).

Primero se mide qué tan bien cada juez reproduce la recomendación del usuario; luego se buscan las reseñas en las que
el juez discrepa con mucha seguridad.

In [ ]:
%%time
# Probabilidad de "recomienda" fuera del fold para todas las reseñas
modelo_e2 = make_pipeline(tfidf(), LogisticRegression(max_iter=2000, class_weight='balanced'))
df['prob_modelo'] = cross_val_predict(modelo_e2, df['review_clean'], df['recomienda'], cv=CV,
                                      method='predict_proba', n_jobs=N_JOBS)[:, 1]

# Puntaje VADER (de -1 a +1) sobre el texto natural
vader = SentimentIntensityAnalyzer()
df['vader'] = df['review_norm'].map(lambda t: vader.polarity_scores(t)['compound'])

In [ ]:
# ¿Que juez reproduce mejor la recomendacion del usuario?
tabla_e2 = pd.DataFrame([
    evaluar('VADER (léxico, sin entrenamiento)', df['recomienda'], (df['vader'] >= 0.05).astype(int), df['vader']),
    evaluar('Regresión Logística (fuera del fold)', df['recomienda'], (df['prob_modelo'] >= 0.5).astype(int), df['prob_modelo']),
]).set_index('Modelo')[['Accuracy', 'F1 macro', 'AUC-ROC']]

# Reseñas candidatas a incoherentes segun cada juez (discrepancia con mucha seguridad)
incoh_vader = ((df['vader'] < -0.5) & (df['recomienda'] == 1)) | ((df['vader'] > 0.5) & (df['recomienda'] == 0))
incoh_modelo = ((df['prob_modelo'] < 0.1) & (df['recomienda'] == 1)) | ((df['prob_modelo'] > 0.9) & (df['recomienda'] == 0))
df['incoherente'] = incoh_vader & incoh_modelo     # ambos jueces coinciden: candidata fuerte

tabla_e2['% marcadas como incoherentes'] = [incoh_vader.mean() * 100, incoh_modelo.mean() * 100]
print(f"Candidatas fuertes (ambos jueces coinciden): {df['incoherente'].sum()} reseñas ({df['incoherente'].mean():.2%})")
tabla_e2.to_csv(f'{DIR_RESULTADOS}/e2_jueces.csv')
tabla_e2

In [ ]:
# Ejemplos de candidatas fuertes
df.loc[df['incoherente'], ['name', 'sentimiento', 'vader', 'prob_modelo', 'review']].sample(8, random_state=1)

**Interpretación:** VADER marca alrededor de 10 % de reseñas como incoherentes, pero muchas son errores del léxico
(jerga como *“this shit is so fire”*, que es un elogio). El modelo entrenado reproduce mucho mejor la recomendación y
solo discrepa con seguridad en menos del 1 % de las reseñas. Las candidatas en las que ambos jueces coinciden son,
sobre todo, reseñas que critican un aspecto puntual pero aun así recomiendan el juego, o que recomiendan por error.

## 8. E3 – ¿La reseña será considerada graciosa?
Objetivo: `funny` = 1 si la reseña recibió al menos un voto de “gracioso” (11.9 %). Es la tarea más difícil, porque el
humor depende del contexto y de la comunidad. Métrica principal: **AUC-PR** (un modelo al azar obtendría 0.119).

In [ ]:
%%time
train_e3, test_e3 = train_test_split(df, test_size=0.2, stratify=df['funny'], random_state=RANDOM_STATE)

modelos_e3 = {
    'Naive Bayes (Complement)': make_pipeline(tfidf(), ComplementNB()),
    'Regresión Logística':      make_pipeline(tfidf(), LogisticRegression(max_iter=2000, class_weight='balanced')),
    'SVM lineal':               make_pipeline(tfidf(), LinearSVC(C=0.3, class_weight='balanced')),
}

resultados_e3 = []
for nombre, modelo in modelos_e3.items():
    modelo.fit(train_e3['review_clean'], train_e3['funny'])
    resultados_e3.append(evaluar(nombre, test_e3['funny'], modelo.predict(test_e3['review_clean']),
                                 puntajes(modelo, test_e3['review_clean'])))

tabla_e3 = pd.DataFrame(resultados_e3).set_index('Modelo').sort_values('AUC-PR', ascending=False)
tabla_e3.to_csv(f'{DIR_RESULTADOS}/e3_resultados.csv')
tabla_e3[['AUC-PR', 'AUC-ROC', 'Precision (clase 1)', 'Recall (clase 1)', 'F1 (clase 1)']]

In [ ]:
# Terminos que el modelo asocia con reseñas graciosas
lr_e3 = modelos_e3['Regresión Logística']
coef_e3 = pd.Series(lr_e3.named_steps['logisticregression'].coef_[0],
                    index=lr_e3.named_steps['tfidfvectorizer'].get_feature_names_out()).nlargest(20)

plt.figure(figsize=(8, 6))
plt.barh(coef_e3.index[::-1], coef_e3.values[::-1], color=C_MAIN, height=0.6)
plt.title('E3 – Términos asociados a reseñas graciosas')
plt.xlabel('Coeficiente (Regresión Logística)')
plt.tight_layout()
guardar_figura('e3_terminos')
plt.show()

## 9. Resumen de resultados

In [ ]:
# Mejor modelo de cada pregunta con su metrica principal
mejor_p2 = tabla_p2.index[0]
mejor_e3 = tabla_e3.index[0]

resumen = pd.DataFrame([
    {'Pregunta': 'P1 – Recomienda', 'Mejor modelo': tabla_p1.index[0], 'Métrica principal': 'F1 macro',
     'Valor': tabla_p1.iloc[0]['F1 macro'], 'Referencia (azar)': 0.5},
    {'Pregunta': 'P2 – Útil', 'Mejor modelo': mejor_p2, 'Métrica principal': 'AUC-PR',
     'Valor': tabla_p2.iloc[0]['AUC-PR'], 'Referencia (azar)': y_test_p2.mean()},
    {'Pregunta': 'P3 – Tema', 'Mejor modelo': f'{algoritmo} (k={K}) + Reg. Logística', 'Métrica principal': 'F1 macro del clasificador',
     'Valor': f1_score(test_p3['tema'], pred_temas, average='macro'), 'Referencia (azar)': 1 / K},
    {'Pregunta': 'E1 – Recomendador', 'Mejor modelo': 'TF-IDF + similitud coseno', 'Métrica principal': 'Hit@5',
     'Valor': tabla_e1.set_index('Métrica').loc['Hit@5', 'Valor'], 'Referencia (azar)': 5 / len(perfil_a)},
    {'Pregunta': 'E2 – Incoherencia', 'Mejor modelo': 'Regresión Logística vs. VADER', 'Métrica principal': 'AUC-ROC del juez',
     'Valor': tabla_e2.loc['Regresión Logística (fuera del fold)', 'AUC-ROC'], 'Referencia (azar)': 0.5},
    {'Pregunta': 'E3 – Graciosa', 'Mejor modelo': mejor_e3, 'Métrica principal': 'AUC-PR',
     'Valor': tabla_e3.iloc[0]['AUC-PR'], 'Referencia (azar)': test_e3['funny'].mean()},
])
resumen.to_csv(f'{DIR_RESULTADOS}/resumen.csv', index=False)
resumen

## 10. Guardado de modelos para la aplicación
La app de Streamlit (`code/app/app.py`) carga estos archivos. Para P1 se guarda la Regresión Logística, porque entrega
probabilidades interpretables y su rendimiento es cercano al del SVM.

In [ ]:
# P2 para la app: una reseña nueva no tiene antiguedad, asi que se usa el modelo sin antiguedad
modelo_app_p2 = lgbm().fit(train_p2[NUM_SIN_ANTIG], y_train_p2)

joblib.dump(modelo_lr_p1, f'{DIR_MODELOS}/p1_regresion_logistica.joblib', compress=3)
joblib.dump({'modelo': modelo_app_p2, 'columnas': NUM_SIN_ANTIG}, f'{DIR_MODELOS}/p2_lightgbm.joblib', compress=3)
joblib.dump({'clasificador': clasificador_temas, 'nombres': nombres_temas, 'palabras': top_temas}, f'{DIR_MODELOS}/p3_temas.joblib', compress=3)
joblib.dump({'nombres': list(nombres_juegos), 'matriz': matriz_perfiles, 'terminos': terminos_reco,
             'similitud': similitud.astype(np.float32)}, f'{DIR_MODELOS}/e1_recomendador.joblib', compress=3)
joblib.dump({'tablas': {'P1': tabla_p1, 'P2': tabla_p2, 'E3': tabla_e3, 'E1': tabla_e1, 'E2': tabla_e2}, 'resumen': resumen},
            f'{DIR_MODELOS}/resultados.joblib', compress=3)

for archivo in sorted(os.listdir(DIR_MODELOS)):
    print(f"{archivo:35s} {os.path.getsize(os.path.join(DIR_MODELOS, archivo)) / 1e6:6.1f} MB")